In [16]:
# TensorFlow and tf.keras 
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import Input, layers
import os

print(tf.__version__)

2.14.0


In [17]:
load_saved_model = False
save_model  = False
EPOCHS = 10
saved_model_path = 'saved_models/cifar_cnn_model'
tfl_file_name = saved_model_path + '.tflite'
use_model_func = True

In [18]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
train_labels = train_labels.squeeze()
test_labels = test_labels.squeeze()

input_shape = train_images.shape[1:]
train_images = train_images / 255.0
test_images = test_images / 255.0
print("Training Images range from {:2.5f} to {:2.5f}".format(np.min(train_images), np.max(train_images)))
print("Test     Images range from {:2.5f} to {:2.5f}".format(np.min(test_images), np.max(test_images)))

Training Images range from 0.00000 to 1.00000
Test     Images range from 0.00000 to 1.00000


In [19]:
def conv_block(num_channels=32, kernel_size=(3,3), pool_size=(2,2),
               padding='same', drop_rate=None, activation='relu',
               use_batchnorm=False):

    def conv_blk_func(inputs):

        x = layers.Conv2D(num_channels, kernel_size=kernel_size,
                          padding=padding)(inputs)

        if use_batchnorm:
            x = layers.BatchNormalization()(x)

        x = layers.Activation(activation)(x)

        if pool_size != (1,1):
            x = layers.MaxPooling2D(pool_size=pool_size)(x)

        if drop_rate is not None:
            x = layers.Dropout(drop_rate)(x)

        return x

    return conv_blk_func

In [20]:
def build_model(input_shape, model_config):

    inputs = Input(shape=input_shape)
    x = inputs

    for idx in range(len(model_config['conv_num_channels'])):

        x = conv_block(
            num_channels=model_config['conv_num_channels'][idx],
            kernel_size=model_config['conv_kernel_sizes'][idx],
            pool_size=model_config['conv_pool_sizes'][idx],
            drop_rate=model_config['conv_drop_rates'][idx],
            use_batchnorm=model_config['conv_use_batchnorm'][idx]
        )(x)

    x = layers.GlobalAveragePooling2D()(x)

    for idx in range(len(model_config['dense_sizes'])):

        x = layers.Dense(model_config['dense_sizes'][idx])(x)

        if model_config['dense_batchnorm'][idx]:
            x = layers.BatchNormalization()(x)

        x = layers.Activation('relu')(x)

        if model_config['dense_droprates'][idx] is not None:
            x = layers.Dropout(model_config['dense_droprates'][idx])(x)

    x = layers.Dense(model_config['num_classes'])(x)

    model = tf.keras.models.Model(inputs=inputs, outputs=x)

    return model

In [21]:
if not load_saved_model:

    model_config = {

        'conv_num_channels':[16,32,64],
        'conv_kernel_sizes':[(3,3),(3,3),(3,3)],
        'conv_pool_sizes':[(2,2),(2,2),(2,2)],
        'conv_drop_rates':[0.2,0.2,0.2],
        'conv_use_batchnorm':[True,True,True],

        'dense_sizes':[64],
        'dense_droprates':[0.3],
        'dense_batchnorm':[True],

        'num_classes':10
    }

    model = build_model(input_shape, model_config)

else:
    model = tf.keras.models.load_model(saved_model_path)

In [22]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv2d_3 (Conv2D)           (None, 32, 32, 16)        448       


                                                                 
 batch_normalization_4 (Bat  (None, 32, 32, 16)        64        
 chNormalization)                                                
                                                                 
 activation_6 (Activation)   (None, 32, 32, 16)        0         
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 16, 16, 16)        0         
 g2D)                                                            
                                                                 
 dropout_5 (Dropout)         (None, 16, 16, 16)        0         
                                                                 
 conv2d_4 (Conv2D)           (None, 16, 16, 32)        4640      
                                                                 
 batch_normalization_5 (Bat  (None, 16, 16, 32)        128       
 chNormalization)                                                
          

In [23]:
if not load_saved_model:

    train_hist = model.fit(
        train_images,
        train_labels,
        validation_data=(test_images,test_labels),
        epochs=EPOCHS
    )

Epoch 1/10
1563/1563 [==============================] - 57s 35ms/step - loss: 1.6831 - accuracy: 0.3823 - val_loss: 1.5618 - val_accuracy: 0.4189
Epoch 2/10
1563/1563 [==============================] - 52s 33ms/step - loss: 1.3896 - accuracy: 0.4941 - val_loss: 1.7977 - val_accuracy: 0.3843
Epoch 3/10
1563/1563 [==============================] - 49s 32ms/step - loss: 1.2873 - accuracy: 0.5414 - val_loss: 1.1211 - val_accuracy: 0.5869
Epoch 4/10
1563/1563 [==============================] - 49s 32ms/step - loss: 1.2223 - accuracy: 0.5624 - val_loss: 1.1341 - val_accuracy: 0.5796
Epoch 5/10
1563/1563 [==============================] - 51s 32ms/step - loss: 1.1729 - accuracy: 0.5833 - val_loss: 1.2687 - val_accuracy: 0.5423
Epoch 6/10
1563/1563 [==============================] - 82s 52ms/step - loss: 1.1357 - accuracy: 0.5956 - val_loss: 1.0332 - val_accuracy: 0.6299
Epoch 7/10
1563/1563 [==============================] - 81s 52ms/step - loss: 1.1122 - accuracy: 0.6043 - val_loss: 0.9211 -

In [24]:
num_calibration_steps = 25
converter = tf.lite.TFLiteConverter.from_keras_model(model)
if True: 

  converter.optimizations = [tf.lite.Optimize.DEFAULT]

  def representative_dataset_gen():
    for i in range(num_calibration_steps):
      next_input = train_images[i:i+1,:,:,:]
      yield [next_input.astype(np.float32)]

  converter.representative_dataset = representative_dataset_gen
  converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

  converter.inference_input_type = tf.int8
  converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open(tfl_file_name, "wb") as fpo:
  fpo.write(tflite_quant_model)
print(f"Wrote to {tfl_file_name}")
!ls -l $tfl_file_name

INFO:tensorflow:Assets written to: C:\Users\vipra\AppData\Local\Temp\tmp935il660\assets


INFO:tensorflow:Assets written to: C:\Users\vipra\AppData\Local\Temp\tmp935il660\assets
c:\tf214_hw2\Lib\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Wrote to saved_models/cifar_cnn_model.tflite


'ls' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path="saved_models/cifar_cnn_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()

scale, zero_point = input_details[0]["quantization"]

print("Input scale:", scale)
print("Input zero point:", zero_point)

Input scale: 0.003921568859368563
Input zero point: -128


In [5]:
from tensorflow.keras.datasets import cifar10

(_, _), (test_images, test_labels) = cifar10.load_data()

test_images = test_images / 255.0
test_labels = test_labels.squeeze()

print("Dataset loaded")
print("Example labels:", test_labels[:10])

Dataset loaded
Example labels: [3 8 8 0 6 6 1 6 3 1]


In [6]:
img1 = test_images[0]
img2 = test_images[1]

label1 = test_labels[0]
label2 = test_labels[1]

print("Input1 true label:", label1)
print("Input2 true label:", label2)

Input1 true label: 3
Input2 true label: 8


In [9]:
H, W, C = input_details[0]['shape'][1:4]  # height, width, channels

img1_resized = tf.image.resize(img1, [H, W]).numpy()
img2_resized = tf.image.resize(img2, [H, W]).numpy()

img1_q = np.round(img1_resized / scale + zero_point).astype(np.int8)
img2_q = np.round(img2_resized / scale + zero_point).astype(np.int8)

img1_flat = img1_q.flatten().astype(np.int8)
img2_flat = img2_q.flatten().astype(np.int8)

print("Height:", H)
print("Width:", W)
print("Channels:", C)
print("Input size:", len(img1_flat))

Height: 32
Width: 32
Channels: 3
Input size: 3072


In [10]:
def write_c_array(f, name, arr):
    f.write(f"const int8_t {name}[{len(arr)}] = {{\n")
    for i, val in enumerate(arr):
        if i % 12 == 0:
            f.write("  ")
        f.write(f"{int(val)}, ")
        if i % 12 == 11:
            f.write("\n")
    f.write("\n};\n\n")

with open("test_inputs.h", "w") as f:
    write_c_array(f, "input1", img1_flat)
    write_c_array(f, "input2", img2_flat)

print("Created file: test_inputs.h")

Created file: test_inputs.h
